<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/12-case-studies-and-capstone/04-deepfake-image-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study D — Deepfake image detection: is a VLM the right tool?

**Goal:** Take a classifier-shaped ask (flag manipulated or AI-generated photos of people before a human trusts them) and settle **which tool to ship** with an eval: a general vision-language model asked zero-shot, versus a dedicated detector. The answer comes from precision/recall at a chosen threshold, not from how convincing the model's explanation sounds.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

> **The thesis, up front:** a VLM will always *answer* "real or fake?", fluently and with reasons. That fluency is the trap. On a detection task the only thing that matters is the error profile (how many fakes slip through, how many real people get falsely flagged), and a general model's confidence is not calibrated for it. This case builds the eval first ([02](../02-evals-basics/01-measuring-outputs.ipynb), [04](../04-evals/01-golden-sets.ipynb)), then lets the numbers decide where the VLM belongs: as a triage assistant for a human reviewer, or not at all.

> **🚧 Skeleton —** structure, scorer and VLM call are written; the cells marked `TODO` (dataset manifest, dedicated detector) still need filling in before this runs end to end. Multimodal is deliberately off the core spine ([curriculum decision #3](../docs/curriculum-plan.md)); this is a *case study* that composes the evals spine, not a new module.

## Setup

Self-contained. Get a free key at [console.groq.com](https://console.groq.com/); in Colab add it via the **key icon** → secret `GROQ_API_KEY`. (Full walkthrough: [00-setup](../00-setup/00-environment.ipynb).)

This case needs an **image-capable** model. Groq's vision docs list `qwen/qwen3.8-27b` (up to 3 images and 20 MB per request, as of 2026-10). If calls 404, check the current list at [console.groq.com/docs/vision](https://console.groq.com/docs/vision).

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git@v0.1.0"

In [ ]:
from aien import setup
client, VISION_MODEL = setup(model="qwen/qwen3.8-27b")

## The task

A trust-and-safety team reviews profile and identity photos. Their ask: *"flag photos of people that were face-swapped or generated, so a reviewer looks before we trust them."* Scoped (the [11/01](../11-customer-craft/01-scoping-and-discovery.ipynb) way), that becomes:

- **In scope:** photorealistic images of people that are fully generated (GAN / diffusion) or locally manipulated (face swap, face reenactment, inpainting).
- **Out of scope:** obvious edits, memes, satire, filters, stickers. A meme is not a deception risk; including it would inflate recall on easy cases and hide the hard ones.
- **Output:** a score in `[0, 1]` (probability "fake") plus a threshold, so the team can choose its own trade between missed fakes and falsely flagged real people.

### Data rules (non-negotiable)

- Use only **public datasets whose license permits this use**, and record the license next to each source.
- **No images of real private individuals** collected ad hoc. Generated faces and consented research datasets only.
- Keep a **held-out split** you never look at while writing prompts, or the eval measures your prompt-tuning, not the model.

### Groups, not just labels

As in [Case B](02-contract-extraction-pipeline-vs-agent.ipynb), an average over easy items hides the result. Every image carries a `tag` naming the kind of difficulty it represents:

| tag | label | what it tests |
|---|---|---|
| `real_clean` | real | camera originals: the false-positive baseline |
| `real_processed` | real | real photos after resize / JPEG / phone filters: does processing alone look "fake"? |
| `fake_gan` | fake | fully generated faces (GAN) |
| `fake_diffusion` | fake | fully generated faces (diffusion) |
| `fake_swap` | fake | face swap / reenactment on a real body and background |
| `fake_laundered` | fake | fakes after screenshot + recompression, the way they actually arrive |

In [ ]:
# TODO: fill from a licensed public dataset. One dict per image.
#   path: local file (download into /content/data/...) or an https URL
#   source / license: recorded per item, so the eval report can cite them
DATASET = [
    # {"id": "R01", "tag": "real_clean",     "label": "real", "path": "...", "source": "...", "license": "..."},
    # {"id": "F01", "tag": "fake_diffusion", "label": "fake", "path": "...", "source": "...", "license": "..."},
]

TAGS = ["real_clean", "real_processed", "fake_gan", "fake_diffusion", "fake_swap", "fake_laundered"]
assert all(d["tag"] in TAGS and d["label"] in ("real", "fake") for d in DATASET)
print(f"{len(DATASET)} images:", {t: sum(d["tag"] == t for d in DATASET) for t in TAGS})

## A shared scorer before any model

Both approaches return a **score**, not a verdict, and the scorer turns scores into the numbers the team actually decides on. Accuracy alone is the wrong headline here: if 2% of uploads are fake, "everything is real" scores 98%.

- **Recall (fake):** of the fakes, how many did we catch? A miss means a fake gets trusted.
- **False-positive rate:** of the real photos, how many did we flag? Each one is a real person sent to review, or worse, rejected.
- **AUC:** threshold-free ranking quality, for comparing approaches before choosing an operating point.

In [ ]:
import base64, io, json, re, urllib.request

def load_bytes(item):
    p = item["path"]
    if p.startswith("http"):
        with urllib.request.urlopen(p, timeout=30) as r:
            return r.read()
    with open(p, "rb") as f:
        return f.read()

def to_data_uri(raw, mime="image/jpeg"):
    return f"data:{mime};base64," + base64.b64encode(raw).decode()

def confusion(rows, threshold):
    tp = sum(r["label"] == "fake" and r["score"] >= threshold for r in rows)
    fn = sum(r["label"] == "fake" and r["score"] < threshold for r in rows)
    fp = sum(r["label"] == "real" and r["score"] >= threshold for r in rows)
    tn = sum(r["label"] == "real" and r["score"] < threshold for r in rows)
    return tp, fn, fp, tn

def metrics(rows, threshold=0.5):
    tp, fn, fp, tn = confusion(rows, threshold)
    return {"recall": tp / max(tp + fn, 1), "fpr": fp / max(fp + tn, 1),
            "precision": tp / max(tp + fp, 1), "n": len(rows)}

def auc(rows):
    # Probability a random fake scores above a random real (ties count half).
    fakes = [r["score"] for r in rows if r["label"] == "fake"]
    reals = [r["score"] for r in rows if r["label"] == "real"]
    if not fakes or not reals:
        return float("nan")
    wins = sum((f > g) + 0.5 * (f == g) for f in fakes for g in reals)
    return wins / (len(fakes) * len(reals))

## Approach 1 — a VLM, zero-shot

The tempting build: send the image to a vision model and ask. We ask for a fixed JSON schema ([01/01 structured output](../01-model-apis/01-structured-output.ipynb)) with a probability, so it plugs into the same scorer as the detector. The `evidence` field is kept for the reviewer UI, but it is **never** scored: a plausible reason is not a correct one.

In [ ]:
MAX_TOKENS = 1024   # headroom for models that reason before answering

VLM_PROMPT = (
    "You are reviewing a photo of a person for signs that it was AI-generated or face-swapped. "
    "Ordinary photo processing (compression, resizing, filters) is NOT manipulation. "
    "Reply with ONLY a JSON object: "
    '{"p_fake": <number 0-1>, "evidence": "<one short sentence>"}'
)

def parse_json(text):
    m = re.search(r"\{.*\}", text or "", re.DOTALL)
    try:
        return json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        return None

def vlm_score(raw):
    resp = client.chat.completions.create(
        model=VISION_MODEL, max_tokens=MAX_TOKENS, temperature=0,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": VLM_PROMPT},
            {"type": "image_url", "image_url": {"url": to_data_uri(raw)}},
        ]}])
    out = parse_json(resp.choices[0].message.content)
    try:
        p = float(out["p_fake"])
    except (TypeError, KeyError, ValueError):
        return None, None                       # unparseable: counted, not guessed
    return min(max(p, 0.0), 1.0), out.get("evidence")

> **⚠️ Production reality —** a generative model's stated probability is a *token it wrote*, not a calibrated estimate. Expect clustering at round numbers (0.1, 0.8, 0.95) and confident, specific-sounding evidence ("asymmetric earrings", "warped background") on images that are genuinely real. Measure the unparseable rate too: every `None` is an image the system silently failed to judge.

## Approach 2 — a dedicated detector

A model trained for exactly this task, which outputs a score directly. This is the comparison that keeps the VLM honest.

Like the GPU appendices in [06](../06-adaptation/01-fine-tune-vs-rag-vs-prompt.ipynb) and [09](../09-serving-inference/01-serving-frameworks.ipynb), it is **fenced and optional**: it may need a GPU runtime and a model download, so the rest of the notebook runs without it.

In [ ]:
DETECTOR_ENABLED = False   # flip to True once detector_score is implemented

def detector_score(raw):
    # TODO: load a published deepfake / AI-image detector (record its license and
    # training data), run it on the image bytes, return P(fake) in [0, 1].
    # Check its training distribution: a detector trained on one generator family
    # often fails on newer ones, which is exactly what the per-tag table exposes.
    raise NotImplementedError("Plug in a dedicated detector, then set DETECTOR_ENABLED = True")

## Run both, several times

One run is an anecdote. Each image goes through each approach `REPEATS` times; the cell resumes where it stopped, so a rate limit only costs a rerun.

> **🔵 Hitting a rate limit?** Image requests are large (the docs list 2,048 tokens per image for this model). If you get a `429`, just **rerun the cell**.

In [ ]:
REPEATS = 2

APPROACHES = {"vlm": vlm_score}
if DETECTOR_ENABLED:
    APPROACHES["detector"] = lambda raw: (detector_score(raw), None)

results = globals().get("results", [])          # rerun-safe: resume instead of starting over
done = {(r["approach"], r["run"], r["id"]) for r in results}

for run in range(REPEATS):
    for d in DATASET:
        raw = None
        for name, fn in APPROACHES.items():
            if (name, run, d["id"]) in done:
                continue
            raw = raw or load_bytes(d)
            score, evidence = fn(raw)
            results.append({"approach": name, "run": run, "id": d["id"], "tag": d["tag"],
                            "label": d["label"], "score": score, "evidence": evidence})
    print(f"run {run + 1}/{REPEATS} done ({len(results)} results)")

## The verdict — same images, measured

In [ ]:
def rows(approach, **where):
    return [r for r in results if r["approach"] == approach
            and all(r[k] == v for k, v in where.items())]

print(f"{'':9} {'AUC':>5} {'recall@.5':>10} {'FPR@.5':>7} {'unparsed':>9}")
for a in APPROACHES:
    rs = rows(a)
    scored = [r for r in rs if r["score"] is not None]
    m = metrics(scored)
    print(f"{a:9} {auc(scored):>5.2f} {m['recall']:>10.0%} {m['fpr']:>7.0%} "
          f"{len(rs) - len(scored):>4}/{len(rs)}")

## Where does each approach fail? By group

Read the real groups for false positives and the fake groups for misses. The interesting rows are `real_processed` (does compression alone trigger a flag?) and `fake_laundered` (does a screenshot erase the signal?).

In [ ]:
print(f"{'group':15} {'n':>3} " + " ".join(f"{a + ' flagged':>16}" for a in APPROACHES))
for t in TAGS:
    n = len({d["id"] for d in DATASET if d["tag"] == t})
    cells_ = []
    for a in APPROACHES:
        scored = [r for r in rows(a, tag=t) if r["score"] is not None]
        flagged = sum(r["score"] >= 0.5 for r in scored) / max(len(scored), 1)
        cells_.append(f"{flagged:>16.0%}")
    print(f"{t:15} {n:>3} " + " ".join(cells_))
# "flagged" is recall on fake_* rows and false-positive rate on real_* rows.

## Choosing the operating point

0.5 is not a law. The team picks a threshold from the cost of each error: a missed fake on an identity check is expensive; a false flag costs a reviewer a minute and the user a delay. A common way to state it: *the highest recall we can get while keeping FPR under X%*.

In [ ]:
TARGET_FPR = 0.05

def threshold_for_fpr(rs, target):
    for t in sorted({r["score"] for r in rs}):
        if metrics(rs, t)["fpr"] <= target:
            return t
    return 1.0

for a in APPROACHES:
    scored = [r for r in rows(a) if r["score"] is not None]
    if not scored:
        continue
    t = threshold_for_fpr(scored, TARGET_FPR)
    m = metrics(scored, t)
    print(f"{a:9} threshold {t:.2f} → recall {m['recall']:.0%} at FPR {m['fpr']:.0%}")

> **🚩 Common mistake —** tuning the threshold on the same images you report. Pick it on one split, report on the held-out one; otherwise the recall above is a best case you'll never see in production.

## Robustness: the laundering test

Fakes rarely arrive as the original file. Re-encode every image at a lower JPEG quality and rescore. A detector that leans on high-frequency artifacts can collapse here; check whether real photos *also* start getting flagged.

In [ ]:
from PIL import Image

def recompress(raw, quality=60, scale=0.75):
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    img = img.resize((int(img.width * scale), int(img.height * scale)))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    return buf.getvalue()

laundered = []
for d in DATASET:
    raw = recompress(load_bytes(d))
    for name, fn in APPROACHES.items():
        score, _ = fn(raw)
        laundered.append({"approach": name, "id": d["id"], "tag": d["tag"],
                          "label": d["label"], "score": score})

for a in APPROACHES:
    before = [r for r in rows(a, run=0) if r["score"] is not None]
    after = [r for r in laundered if r["approach"] == a and r["score"] is not None]
    mb, ma = metrics(before), metrics(after)
    print(f"{a:9} recall {mb['recall']:.0%} → {ma['recall']:.0%}   FPR {mb['fpr']:.0%} → {ma['fpr']:.0%}")

## Where the VLM *does* belong

Whatever the numbers say about the VLM as a classifier, it can still earn a place in the system:

- **Triage, not verdict.** The detector scores; images in the uncertain band go to a human, and the VLM's `evidence` gives the reviewer a place to start looking.
- **Context the pixels lack.** "Does the caption claim this is a news photo from an event?" is a language question a detector can't answer.
- **Policy, not perception.** Deciding whether a flagged image violates *this team's* rules (satire allowed, impersonation not) is text reasoning over the detector's output.

> **⭐ Key takeaway —** a model that answers every question is not a model that should decide every question. On a detection task, build the scorer first, compare against a purpose-built baseline, pick the threshold from the cost of each error, and test on the inputs as they actually arrive. Then decide where the general model fits; often it's next to the classifier, not instead of it.

> **🔵 Interview signal —** "we measured the VLM against a dedicated detector at a fixed FPR, per difficulty group and after recompression, and used the VLM only for reviewer triage" shows the same judgment as [Case B](02-contract-extraction-pipeline-vs-agent.ipynb): knowing when *not* to reach for the general tool.

## Exercises

1. **Fill the skeleton.** Build `DATASET` from a licensed public source, at least 10 images per tag, with a held-out split. Record every license in the manifest.
2. **Calibration.** Bucket the VLM's `p_fake` into bins (0–0.2, 0.2–0.4, …) and plot the observed fake rate per bin. Is a stated 0.9 actually fake 90% of the time?
3. **Prompt vs reality.** Rewrite `VLM_PROMPT` with a few-shot description of known artifacts. Does AUC move on the *held-out* split, or only on the images you looked at while writing it?
4. **The cost at scale.** At 1M uploads/month and 2% fakes, compute the number of missed fakes and false flags per month at your chosen threshold for each approach. Write the one-sentence design-doc justification for what you'd ship.
5. **Hybrid.** Send only images whose detector score falls in an uncertain band (say 0.3–0.7) to the VLM plus a human. What share of traffic needs review, and what recall do you get overall?